# Kapitel 5 — Klassifizieren und entscheiden · Notebook zum Ausfüllen

*Mathe für KI, Schritt für Schritt*, Belk Siri — Begleit-Notebook.

[In Google Colab öffnen](https://colab.research.google.com/github/belksiri/books/blob/main/maths-pour-l-ia-pas-a-pas/notebooks/de/ch05_a_completer.ipynb) · Begleitseite: https://belksiri.github.io/books/maths-pour-l-ia-pas-a-pas/de.html

Die zu bearbeitenden Zellen sind mit **ZU ERGÄNZEN** gekennzeichnet. Führen Sie die Zellen der Reihe nach aus: Solange eine Antwort fehlt, melden die Überprüfungen dies ohne Fehler. Die Lösung finden Sie in `ch05_corrige.ipynb` auf derselben Begleitseite.

**Inhalt.** Daten · Die Berechnungen der Lektionen 5.1 bis 5.5 · Aufgaben 5.1 bis 5.6

**Konventionen des Notebooks.**
- Entscheidungsregel (Gleichung 5.1): Eine Nachricht wird markiert, wenn ihre geschätzte Wahrscheinlichkeit **größer oder gleich** dem Schwellenwert ist. Für die zwölf Nachrichten wird die Regel auf den **nicht gerundeten** Wert `p_exact` angewendet, der mit der Sigmoidfunktion neu berechnet wird; die Tabelle im Buch zeigt nur den auf zwei Dezimalstellen gerundeten Wert. In den Aufgaben, in denen die Wahrscheinlichkeiten als Dezimalzahlen gegeben sind, sind diese Dezimalzahlen die zu verwendenden Werte.
- Ein Maß, dessen Nenner null ist (zum Beispiel die Präzision, wenn kein Alarm ausgelöst wird), ist `None` und wird als „undefiniert“ angezeigt.
- Der F1 wird als $2\,VP / (2\,VP + FP + FN)$ berechnet (Gleichung 5.8). Der Code behält die französischen Abkürzungen: VP = richtig positiv, FP = falsch positiv, FN = falsch negativ, VN = richtig negativ.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Farben des Buchs: eine Farbe = eine Rolle
SPAM, LEGITIME, SIGNALE, DENOM = '#d62728', '#b9b6ae', '#1f4e8c', '#ffd54f'
SEUIL, ORANGE, BLEU, VERT = '#8e24aa', '#eb6834', '#2a78d6', '#2e8b57'   # Schwellenwert; Sigmoidfunktion; Verlust; Filter B


def sigmoide(z):
    return 1 / (1 + np.exp(-np.asarray(z, dtype=float)))


def affiche(v):
    """Konvention des Notebooks: Ein Maß mit Nenner null ist None und wird als „undefiniert“ angezeigt."""
    return 'undefiniert' if v is None else f'{v:.2f}'

## Daten

Die Kennungen D5.1, D5.2, D5.E1, D5.E2 und D5.E3 sind die des Buchs und der CSV-Dateien.

Variablen- und Spaltennamen im Code bleiben französisch: `douze` / `dix` / `quatre` (zwölf / zehn / vier Nachrichten), `etiquette` (tatsächliches Label: `'spam'` oder `'legitime'`, legitim), `liens` (Links), `mots_suspects` (verdächtige Wörter), `score` (geschätzte Wahrscheinlichkeit), `calibration` (Filter A), `calibration_b` (Filter B), `tranche_min` / `tranche_max` (Intervallgrenzen), `proba_moyenne` (mittlere vorhergesagte Wahrscheinlichkeit), `p_filtre_A` / `p_filtre_B` (Wahrscheinlichkeiten der Filter A und B), `p_exact` (nicht gerundete Wahrscheinlichkeit), `est_spam` (ist Spam), `seuil` (Schwellenwert), `signalé` (markiert), `perte` (Verlust), `cout` (Kosten), `rappel` (Trefferquote).

In [ ]:
# Daten von Kapitel 5, aus den CSV-Dateien des Buchs übernommen (nicht ändern)

# D5.1 — zwölf Nachrichten: tatsächliches Label, Links, verdächtige Wörter, geschätzte Wahrscheinlichkeit auf zwei Dezimalstellen GERUNDET
douze = pd.DataFrame([
    ('M1', 'spam', 5, 4, 0.95),
    ('M2', 'legitime', 2, 0, 0.12),
    ('M3', 'legitime', 4, 0, 0.50),
    ('M4', 'spam', 3, 3, 0.62),
    ('M5', 'legitime', 1, 0, 0.05),
    ('M6', 'legitime', 2, 2, 0.27),
    ('M7', 'spam', 2, 3, 0.38),
    ('M8', 'legitime', 2, 1, 0.18),
    ('M9', 'spam', 4, 4, 0.88),
    ('M10', 'legitime', 1, 1, 0.08),
    ('M11', 'legitime', 0, 0, 0.02),
    ('M12', 'legitime', 0, 1, 0.03)
], columns=['message', 'etiquette', 'liens', 'mots_suspects', 'score'])

# D5.2 — Filter A: 1000 Nachrichten nach Intervall (Anzahl, Spams, mittlere vorhergesagte Wahrscheinlichkeit)
calibration = pd.DataFrame([
    (0.0, 0.1, 540, 22, 0.03),
    (0.1, 0.3, 150, 29, 0.19),
    (0.3, 0.5, 80, 33, 0.40),
    (0.5, 0.7, 60, 35, 0.60),
    (0.7, 0.9, 90, 71, 0.80),
    (0.9, 1.0, 80, 77, 0.95)
], columns=['tranche_min', 'tranche_max', 'messages', 'spams', 'proba_moyenne'])

# D5.E1 — Filter B: gleiche Tabelle (Aufgabe 5.1)
calibration_b = pd.DataFrame([
    (0.0, 0.1, 400, 40, 0.02),
    (0.1, 0.3, 175, 42, 0.18),
    (0.3, 0.5, 100, 43, 0.41),
    (0.5, 0.7, 100, 55, 0.61),
    (0.7, 0.9, 150, 93, 0.82),
    (0.9, 1.0, 75, 56, 0.97)
], columns=['tranche_min', 'tranche_max', 'messages', 'spams', 'proba_moyenne'])

# D5.E2 — zehn Nachrichten des Vereins (Aufgabe 5.2): Diese Dezimalzahlen sind die zu verwendenden Werte
dix = pd.DataFrame([
    ('N1', 'spam', 0.94),
    ('N2', 'legitime', 0.07),
    ('N3', 'legitime', 0.66),
    ('N4', 'spam', 0.50),
    ('N5', 'legitime', 0.30),
    ('N6', 'spam', 0.21),
    ('N7', 'legitime', 0.42),
    ('N8', 'spam', 0.77),
    ('N9', 'legitime', 0.13),
    ('N10', 'spam', 0.58)
], columns=['message', 'etiquette', 'score'])

# D5.E3 — vier Nachrichten, von zwei Filtern bewertet (Aufgabe 5.3)
quatre = pd.DataFrame([
    ('P1', 'spam', 0.80, 0.99),
    ('P2', 'spam', 0.60, 0.90),
    ('P3', 'legitime', 0.40, 0.10),
    ('P4', 'legitime', 0.20, 0.01)
], columns=['message', 'etiquette', 'p_filtre_A', 'p_filtre_B'])

## Die Berechnungen der Lektionen, in Python

Diese Zellen wiederholen die Berechnungen der Lektionen 5.1 bis 5.5.

In [ ]:
# Exakte Werte der zwölf Nachrichten (Gleichungen 5.3 und 5.4): Auf sie wird die Regel angewendet
w, b = np.array([1, 0.5]), -4
X = douze[['liens', 'mots_suspects']].to_numpy()
z = X @ w + b
p_exact = sigmoide(z)
est_spam = (douze['etiquette'] == 'spam').to_numpy()
y = est_spam.astype(int)                 # 1 = Spam, 0 = legitim

# Lektion 5.1 — den Schwellenwert verschieben (Gleichungen 5.1 und 5.2)
for seuil in (0.5, 0.25):
    signale = p_exact >= seuil                    # Regel: größer ODER GLEICH, auf den exakten Wert
    assert (signale == (douze['score'] >= seuil)).all()   # hier ändert die Rundung keine Entscheidung
    vp, fn = int((signale & est_spam).sum()), int((~signale & est_spam).sum())
    fp, vn = int((signale & ~est_spam).sum()), int((~signale & ~est_spam).sum())
    print(f'Schwellenwert {seuil}: VP = {vp}, FN = {fn}, FP = {fp}, VN = {vn} | '
          f'Trefferquote = {vp}/{vp + fn} = {vp / (vp + fn):.3f}, '            # Nenner: die Zeile der Spams
          f'Fehlalarme = {fp}/{fp + vn} = {fp / (fp + vn):.3f}, '   # Nenner: die Zeile der legitimen
          f'Präzision = {vp}/{vp + fp} = {vp / (vp + fp):.3f}')         # Nenner: die Spalte der markierten
bascule = douze['message'][(p_exact >= 0.25) & (p_exact < 0.5)]
print('Nachrichten, die von 0,5 zu 0,25 wechseln:', list(bascule))

fig, axes = plt.subplots(2, 1, figsize=(7, 3.4), sharex=True)
for ax, seuil in zip(axes, (0.5, 0.25)):
    for q, s in zip(p_exact, est_spam):
        sig = q >= seuil
        ax.scatter(q, 1 if s else 0, s=70, marker='s' if s else 'o', color=SPAM if s else LEGITIME,
                   edgecolor=SIGNALE if sig else 'gray', linewidth=2.4 if sig else 0.6)
    ax.axvline(seuil, color=SEUIL, linewidth=2.5)
    ax.set_yticks([0, 1], ['legitim', 'Spams']); ax.set_ylim(-0.6, 1.6)
    ax.set_title(f'Schwellenwert {seuil}', loc='left', color=SEUIL)
axes[1].set_xlabel('Geschätzte Wahrscheinlichkeit (exakter Wert)'); plt.tight_layout(); plt.show()

In [ ]:
# Lektion 5.2 — Rohwert, geschätzte Wahrscheinlichkeit, Entscheidung; der Fall M3; Kalibrierung
tableau = pd.DataFrame({'message': douze['message'], 'z': z, 'p_exact': p_exact.round(4),
                        'p arrondie': p_exact.round(2),
                        'décision (seuil 0,5)': np.where(p_exact >= 0.5, 'signalé', 'non')})
print(tableau.to_string(index=False))
print('gerundet = Werte aus Lektion 5.1:', np.allclose(p_exact.round(2), douze['score']))
print(f"M3: z = {z[2]}, e^0 = {np.exp(0.0)}, p̂ = 1/(1 + 1) = {p_exact[2]} genau → "
      f"{'markiert' if p_exact[2] >= 0.5 else 'nicht markiert'} beim Schwellenwert 0,5")
print(f'M4: e^(-0,5) = {np.exp(-0.5):.4f}, Sigmoid(0,5) = {sigmoide(0.5):.3f}; M7: Sigmoid(-0,5) = {sigmoide(-0.5):.3f}')
print('Schwellenwert 0,5 für p̂ ⇔ Schwellenwert 0 für z:', bool(((p_exact >= 0.5) == (z >= 0)).all()),
      f'| Schwellenwert 0,25 für p̂ ⇔ z ≥ {np.log(0.25 / 0.75):.2f}')

zs = np.linspace(-5, 4.5, 300)
plt.figure(figsize=(6, 3))
plt.plot(zs, sigmoide(zs), color=ORANGE, linewidth=2.2)
plt.axhline(0.5, color=SEUIL, linewidth=2)
plt.scatter(z[est_spam], p_exact[est_spam], marker='s', color=SPAM, zorder=3, label='Spams')
plt.scatter(z[~est_spam], p_exact[~est_spam], marker='o', color=LEGITIME, edgecolor='gray', zorder=3, label='legitim')
plt.xlabel('Rohwert z'); plt.ylabel('Geschätzte Wahrscheinlichkeit p̂'); plt.legend(); plt.grid(alpha=0.4); plt.show()

# Eine geschätzte Wahrscheinlichkeit lässt sich prüfen: nebeneinander mittleres vorhergesagtes p̂, beobachtete Häufigkeit, Anzahl (Nenner)
cal = calibration.copy()
cal['fréquence observée'] = (cal['spams'] / cal['messages']).round(2)
print(cal[['tranche_min', 'tranche_max', 'proba_moyenne', 'fréquence observée', 'messages', 'spams']].to_string(index=False))
t = cal.iloc[4]
print(f"Intervall 0,7–0,9: vorhergesagt {t['proba_moyenne']:.2f}, beobachtet {int(t['spams'])} / {int(t['messages'])} ≈ "
      f"{t['spams'] / t['messages']:.2f} | erwartet {t['proba_moyenne'] * t['messages']:.0f} Spams, "
      f"Standardabweichung {np.sqrt(t['messages'] * 0.8 * 0.2):.1f}")
print('größte Abweichung zwischen vorhergesagt und beobachtet:', f"{np.max(np.abs(cal['proba_moyenne'] - cal['spams'] / cal['messages'])):.3f}")

In [ ]:
# Lektion 5.3 — die Strafe für einen echten Spam: −ln(p̂) (Gleichung 5.5). np.log ist der NATÜRLICHE Logarithmus.
for p in (0.90, 0.50, 0.10, 0.01):
    print(f'p̂ = {p:.2f} → Strafe = {-np.log(p):.3f}')
print(f'Verhältnisse: {np.log(0.01) / np.log(0.5):.1f} mal die Variante mit 0,50, '
      f'{np.log(0.01) / np.log(0.9):.1f} mal die Variante mit 0,90')
p_spams = p_exact[est_spam]                             # NICHT gerundete Werte aus Lektion 5.2 (M4: 0,622)
penalites = -np.log(p_spams)
print('Spams', list(douze.loc[est_spam, 'message']), ': Strafen', penalites.round(2), f'| Mittelwert = {penalites.mean():.2f}')
print(f'Produkt der Wahrscheinlichkeiten = {p_spams.prod():.3f}; −ln(Produkt) = {-np.log(p_spams.prod()):.2f} '
      f'= Summe der Strafen = {penalites.sum():.2f}')
ps = np.linspace(0.006, 1, 300)
plt.figure(figsize=(6, 3))
plt.plot(ps, -np.log(ps), color=BLEU, linewidth=2.2)
plt.scatter([0.9, 0.5, 0.01], -np.log([0.9, 0.5, 0.01]), marker='s', color=SPAM, zorder=3)
plt.xlabel('Geschätzte Wahrscheinlichkeit p̂ für einen echten Spam'); plt.ylabel('Strafe −ln(p̂)'); plt.grid(alpha=0.4); plt.show()

In [ ]:
# Lektion 5.4 — der Verlust jeder Nachricht (Gleichung 5.6) und der mittlere Verlust (Gleichung 5.7)
def perte(y, p):
    return -(y * np.log(p) + (1 - y) * np.log(1 - p))   # −ln der Wahrscheinlichkeit, die der richtigen Antwort gegeben wird

p_tab = douze['score'].to_numpy()                       # in der Tabelle angezeigte Werte (auf zwei Dezimalstellen)
pertes = perte(y, p_exact)                              # Verluste aus den NICHT gerundeten Werten berechnet, wie im Buch
print(pd.DataFrame({'message': douze['message'], 'y': y, 'p̂': p_tab,
                    'bonne réponse': np.where(y == 1, p_tab, 1 - p_tab), 'perte': pertes.round(2)}).to_string(index=False))
print(f'M4: −ln({p_exact[3]:.3f}) = {pertes[3]:.3f} ≈ {pertes[3]:.2f} (mit dem gerundeten 0,62 erhielte man {-np.log(0.62):.2f})')
print(f'M3: −ln(1 − 0,50) = {-np.log(0.5):.3f}; M6: −ln(1 − 0,27) = {-np.log(0.73):.3f}; M11: {-np.log(0.98):.3f}')
print(f'Summe der gerundeten Verluste = {pertes.round(2).sum():.2f}; mittlerer Verlust = {pertes.mean():.2f} '
      f'(ungerundete Summe: {pertes.sum():.3f})')

# Zuerst anpassen: der mittlere Verlust als Funktion von b (w fest), wie die Verlustkurve aus Kapitel 3
def perte_b(bb):
    return perte(y, sigmoide(X @ w + bb)).mean()
for bb in (-5, -4, -3):
    print(f'b = {bb}: mittlerer Verlust = {perte_b(bb):.2f}')
bs = np.linspace(-7, -1, 6001)
Lb = [perte_b(bb) for bb in bs]
print(f'Tiefpunkt der Kurve: b ≈ {bs[int(np.argmin(Lb))]:.2f}, Verlust {min(Lb):.4f} (gegenüber {perte_b(-4):.4f} für b = −4)')
plt.figure(figsize=(6, 3))
plt.plot(bs, Lb, color=BLEU, linewidth=2.2)
plt.scatter([-4], [perte_b(-4)], marker='D', color=ORANGE, zorder=3)
plt.xlabel('Konstante b'); plt.ylabel('Mittlerer Verlust'); plt.grid(alpha=0.4); plt.show()
print('Quadratische Abweichung für den mit 0,01 bewerteten Spam:', round((1 - 0.01) ** 2, 2), '; für 0,50:', (1 - 0.5) ** 2)

In [ ]:
# Lektion 5.5 — Schwellenwerte vergleichen: Alarme, Fehlalarme, verpasste, Präzision, Trefferquote, F1, Kosten (k = 3)
lignes = []
for seuil in (0.25, 0.3, 0.5, 0.6, 0.7, 0.96):
    sig = p_exact >= seuil                              # exakte Werte
    vp, fp = int((sig & est_spam).sum()), int((sig & ~est_spam).sum())
    fn = int((~sig & est_spam).sum())
    precision = None if vp + fp == 0 else vp / (vp + fp)   # Spalte der markierten leer → undefiniert
    lignes.append({'seuil': seuil, 'alertes': vp + fp, 'fausses alertes': fp, 'manqués': fn,
                   'précision': affiche(precision), 'rappel': f'{vp / (vp + fn):.2f}',
                   'F1': f'{2 * vp / (2 * vp + fp + fn):.2f}', 'coût (k = 3)': fn + 3 * fp})
print(pd.DataFrame(lignes).to_string(index=False))
print(f'F1 beim Schwellenwert 0,3: 2 × 0,8 × 1 / 1,8 = {2 * 0.8 * 1 / 1.8:.2f}; beim Schwellenwert 0,6: 1,5 / 1,75 = {1.5 / 1.75:.2f}')

# Dieselbe Berechnung mit scikit-learn, falls installiert: Seine Konvention für eine undefinierte Präzision ist 0
try:
    from sklearn.metrics import precision_score, f1_score
    predit = (p_exact >= 0.96).astype(int)
    print('scikit-learn beim Schwellenwert 0,96: Präzision =', precision_score(y, predit, zero_division=0),
          '(Konvention zero_division=0); F1 =', f1_score(y, predit, zero_division=0))
except ImportError:
    print('scikit-learn nicht installiert: Diese Berechnung ist optional.')

## Aufgabe 5.1 — Ein Diagramm lesen: zwei Filter im Vergleich zur Diagonale

*Begriffe: geschätzte Wahrscheinlichkeit und Kalibrierung (Lektion 5.2).* Lesen Sie zuerst die Abbildung im Buch. Filter A: Daten D5.2; Filter B: Daten D5.E1.

### a) bis d) Ihre Ablesungen

*Ihre Antworten:* …

In [ ]:
# ZU ERGÄNZEN: e) beobachtete Häufigkeiten von Filter B (Spams / Anzahl in jedem Intervall),
# dann die erwartete Anzahl Spams im Intervall 0,9–1, wenn B kalibriert wäre (mittleres p̂ × Anzahl)
freq_b = None
attendu_derniere = None

In [ ]:
# Überprüfung von e)
if freq_b is None or attendu_derniere is None:
    print('noch nicht ausgefüllt. Ergänzen Sie die vorherige Zelle.')
else:
    print('Häufigkeiten von B:', np.round(freq_b, 2), '→',
          'richtig' if np.allclose(freq_b, [0.10, 0.24, 0.43, 0.55, 0.62, 56 / 75]) else 'bitte prüfen')
    n, q = calibration_b['messages'].iloc[-1], calibration_b['proba_moyenne'].iloc[-1]
    et = np.sqrt(n * q * (1 - q))                       # Standardabweichung der Anzahl Spams (Aufgabe 4.6)
    observe = calibration_b['spams'].iloc[-1]
    print(f'Intervall 0,9–1: erwartet {attendu_derniere:.2f} Spams, beobachtet {observe}, Standardabweichung {et:.2f}, '
          f'Abweichung = {(attendu_derniere - observe) / et:.0f} Standardabweichungen →',
          'richtig' if abs(attendu_derniere - 72.75) < 0.01 else 'bitte prüfen')
    plt.figure(figsize=(4.2, 4.2))
    plt.plot([0, 1], [0, 1], ':', color='gray', label='perfekte Kalibrierung')
    plt.plot(calibration['proba_moyenne'], calibration['spams'] / calibration['messages'], 'o-', color=BLEU, label='Filter A')
    plt.plot(calibration_b['proba_moyenne'], freq_b, '^--', color=VERT, label='Filter B')
    plt.xlabel('Mittlere vorhergesagte Wahrscheinlichkeit'); plt.ylabel('Beobachtete Häufigkeit'); plt.legend(); plt.grid(alpha=0.4)
    plt.show()

### e) Ihre Interpretation

*Ihre Antwort:* …

## Aufgabe 5.2 — Drei Schwellenwerte für einen Verein

*Begriffe: Entscheidungsregel und Konfusionsmatrix (Lektion 5.1), Trefferquote (Recall), Fehlalarmrate und Präzision (Gleichung 5.2), F1 und Kosten (Lektion 5.5).* Zehn vom Sekretariat geprüfte Nachrichten (Daten D5.E2), drei Schwellenwerte: 0,3, 0,5 und 0,7. **Die Wahrscheinlichkeiten in der Tabelle sind unverändert zu verwenden.** Bearbeiten Sie zuerst die Fragen a) bis f) auf Papier.

### a) bis c) Auf Papier

*Ihre Matrizen und Berechnungen:* …

### b) Die Konfusionsmatrix in Python

In [ ]:
# ZU ERGÄNZEN: die vier Felder der Konfusionsmatrix für einen gegebenen Schwellenwert
def matrice_confusion(scores, est_spam, seuil):
    """scores und est_spam: zwei Listen gleicher Länge. Gibt das Wörterbuch
    {'VP': ..., 'FN': ..., 'FP': ..., 'VN': ...} zurück. Regel: markiert, wenn score >= seuil."""
    return None   # ZU ERGÄNZEN

In [ ]:
# Überprüfung: zuerst an den zwölf Nachrichten (EXAKTE Werte), dann an den zehn Nachrichten der Aufgabe
cas = [('Lektion, Schwellenwert 0,5', list(p_exact), list(est_spam), 0.5, (3, 1, 1, 7)),
       ('Lektion, Schwellenwert 0,25', list(p_exact), list(est_spam), 0.25, (4, 0, 2, 6)),
       ('Aufgabe, Schwellenwert 0,7', list(dix['score']), list(dix['etiquette'] == 'spam'), 0.7, (2, 3, 0, 5)),
       ('Aufgabe, Schwellenwert 0,5', list(dix['score']), list(dix['etiquette'] == 'spam'), 0.5, (4, 1, 1, 4)),
       ('Aufgabe, Schwellenwert 0,3', list(dix['score']), list(dix['etiquette'] == 'spam'), 0.3, (4, 1, 3, 2))]
for nom, sc, sp, seuil, attendu in cas:
    m = matrice_confusion(sc, sp, seuil)
    if m is None:
        print(f'{nom}: noch nicht ausgefüllt. Ergänzen Sie die Funktion.')
    else:
        obtenu = (m['VP'], m['FN'], m['FP'], m['VN'])
        print(f'{nom}: VP, FN, FP, VN = {obtenu} →', 'richtig' if obtenu == attendu else 'bitte prüfen')

### c) und e) Die Maße und der F1

In [ ]:
# ZU ERGÄNZEN: Trefferquote, Fehlalarmrate, Präzision (Gleichung 5.2) und F1 (Gleichung 5.8).
# Geben Sie None für ein Maß zurück, dessen Nenner null ist (Konvention des Notebooks).
def mesures(m):
    rappel = None                 # ZU ERGÄNZEN: Nenner = Zeile der Spams
    taux_fausses_alertes = None   # ZU ERGÄNZEN: Nenner = Zeile der legitimen Nachrichten
    precision = None              # ZU ERGÄNZEN: Nenner = Spalte der markierten
    f1 = None                     # ZU ERGÄNZEN: 2 VP / (2 VP + FP + FN)
    return rappel, taux_fausses_alertes, precision, f1

In [ ]:
# Überprüfung von c) und e)
scores_e2, spams_e2 = list(dix['score']), list(dix['etiquette'] == 'spam')
if matrice_confusion(scores_e2, spams_e2, 0.5) is None:
    print('Ergänzen Sie zuerst matrice_confusion.')
elif mesures(matrice_confusion(scores_e2, spams_e2, 0.5))[0] is None:
    print('Maße: noch nicht ausgefüllt. Ergänzen Sie die Funktion mesures.')
else:
    attendus = {0.7: (0.4, 0.0, 1.0), 0.5: (0.8, 0.2, 0.8), 0.3: (0.8, 0.6, 4 / 7)}
    for seuil, att in attendus.items():
        r, f, p, _ = mesures(matrice_confusion(scores_e2, spams_e2, seuil))
        print(f'Schwellenwert {seuil}: Trefferquote = {affiche(r)}, Fehlalarme = {affiche(f)}, Präzision = {affiche(p)} →',
              'richtig' if np.allclose((r, f, p), att) else 'bitte prüfen')
    r, f, p, f1 = mesures(matrice_confusion(scores_e2, spams_e2, 0.95))
    print(f'Schwellenwert 0.95: kein Alarm; Trefferquote = {affiche(r)}, Präzision = {affiche(p)}, F1 = {affiche(f1)} →',
          'richtig' if r == 0 and p is None and f1 == 0 else 'bitte prüfen')
    f1s = [mesures(matrice_confusion(scores_e2, spams_e2, s))[3] for s in (0.7, 0.5, 0.3)]
    print('F1 bei den Schwellenwerten 0,7 / 0,5 / 0,3:', [round(v, 2) for v in f1s], '→',
          'richtig' if np.allclose(f1s, [4 / 7, 0.8, 8 / 12]) else 'bitte prüfen')

### f) Die Kosten der Fehler

In [ ]:
# ZU ERGÄNZEN: Gesamtkosten = verpasste Spams + k × Fehlalarme (Gleichung 5.9)
def cout(m, k):
    return None   # ZU ERGÄNZEN

In [ ]:
# Überprüfung von f)
if matrice_confusion(scores_e2, spams_e2, 0.5) is None or cout({'FN': 0, 'FP': 0}, 3) is None:
    print('Kosten: noch nicht ausgefüllt. Ergänzen Sie die Funktionen.')
else:
    for k, attendu in ((3, [3, 4, 10]), (1, [3, 2, 4])):
        couts = [cout(matrice_confusion(scores_e2, spams_e2, s), k) for s in (0.7, 0.5, 0.3)]
        meilleur = (0.7, 0.5, 0.3)[couts.index(min(couts))]
        print(f'k = {k}: Kosten bei den Schwellenwerten 0,7 / 0,5 / 0,3 = {couts}, bester Schwellenwert {meilleur} →',
              'richtig' if couts == attendu else 'bitte prüfen')

### g) Alle Werte des Schwellenwerts

In [ ]:
# Trefferquote, Präzision und F1 für die Schwellenwerte 0, 0,01, …, 1 (undefinierte Präzision: kein Punkt)
if matrice_confusion(scores_e2, spams_e2, 0.5) is None or mesures(matrice_confusion(scores_e2, spams_e2, 0.5))[0] is None:
    print('Ergänzen Sie matrice_confusion und mesures, um die Kurven zu zeichnen.')
else:
    seuils = np.round(np.arange(0, 1.001, 0.01), 2)
    R, F, P, F1 = zip(*[mesures(matrice_confusion(scores_e2, spams_e2, s)) for s in seuils])
    P = [np.nan if p is None else p for p in P]
    plt.figure(figsize=(6, 3.2))
    plt.step(seuils, R, where='post', color=SPAM, linewidth=2, label='Trefferquote (Zeile der Spams)')
    plt.step(seuils, P, where='post', color=SIGNALE, linewidth=2, linestyle='--', label='Präzision (Spalte der markierten)')
    plt.step(seuils, F1, where='post', color='black', linewidth=1.5, linestyle=':', label='F1')
    for s in (0.3, 0.5, 0.7):
        plt.axvline(s, color=SEUIL, linewidth=1, alpha=0.6)
    plt.xlabel('Schwellenwert'); plt.ylim(-0.05, 1.05); plt.legend(fontsize=8); plt.grid(alpha=0.4); plt.show()

### d) bis f) Ihre Interpretationen

*Ihre Antwort zu d):* …

*Ihr F1 in e):* …

*Ihre Wahl in f):* …

## Aufgabe 5.3 — Interpretieren: gleiche Entscheidungen, unterschiedliche Verluste

*Begriffe: logarithmischer Verlust (Lektionen 5.3 und 5.4), Entscheidungsregel (Gleichung 5.1).* Daten D5.E3: Diese Wahrscheinlichkeiten sind die zu verwendenden Werte.

### a) bis d) Auf Papier

*Ihre Antworten:* …

In [ ]:
# ZU ERGÄNZEN: mittlerer logarithmischer Verlust (Gleichungen 5.6 und 5.7)
def perte_moyenne(etiquettes, probas):
    """etiquettes: Liste aus 1 (Spam) und 0 (legitim); probas: geschätzte Wahrscheinlichkeiten, Spam zu sein."""
    return None   # ZU ERGÄNZEN; np.log ist der natürliche Logarithmus

In [ ]:
# Überprüfung
y4 = list((quatre['etiquette'] == 'spam').astype(int))
if perte_moyenne(y4, quatre['p_filtre_A']) is None:
    print('noch nicht ausgefüllt. Ergänzen Sie die Funktion.')
else:
    p_c = [0.99, 0.01, 0.10, 0.01]
    for nom, probas, att in (('A', quatre['p_filtre_A'], 0.3670), ('B', quatre['p_filtre_B'], 0.0577), ('C', p_c, 1.1827)):
        v = perte_moyenne(y4, probas)
        print(f'Filter {nom}: mittlerer Verlust = {v:.4f} →', 'richtig' if abs(v - att) < 0.0001 else 'bitte prüfen')
    print('Kontrolle mit den zwölf Nachrichten aus Lektion 5.4:', round(perte_moyenne(list(y), list(p_exact)), 2))

## Aufgabe 5.4 — Die Kosten bestimmen den Schwellenwert

*Begriffe: Kosten eines Schwellenwerts (Gleichung 5.9), F1 (Gleichung 5.8).* Wir nehmen wieder die zwölf Nachrichten mit ihren **exakten** Wahrscheinlichkeiten `p_exact`: Auf sie wird die Regel angewendet.

### a) Ihre Vorhersage

*Wenn ein Fehlalarm immer teurer wird, dann wird der beste Schwellenwert…* …

In [ ]:
# ZU ERGÄNZEN: Kosten eines Schwellenwerts für die zwölf Nachrichten (exakte Werte), k = Preis eines Fehlalarms
def cout_total(seuil, k):
    return None   # ZU ERGÄNZEN: verpasste Spams + k × Fehlalarme; verwenden Sie p_exact und est_spam

In [ ]:
# b) bis e) Die besten Schwellenwerte für jedes k, dann die Schwellenwerte des F1
seuils = np.round(np.arange(0, 1.001, 0.01), 2)
if cout_total(0.5, 3) is None:
    print('noch nicht ausgefüllt. Ergänzen Sie die Funktion.')
else:
    attendus = {0.1: (0.27, 0.37), 1: (0.27, 0.62), 3: (0.51, 0.62), 10: (0.51, 0.62)}
    plt.figure(figsize=(6, 3.2))
    for k, att in attendus.items():
        c = np.array([cout_total(s, k) for s in seuils])
        meilleurs = seuils[np.isclose(c, c.min())]
        print(f'k = {k:>4}: minimale Kosten {c.min():.1f}, erreicht für die Schwellenwerte von {meilleurs.min():.2f} bis {meilleurs.max():.2f} →',
              'richtig' if (meilleurs.min(), meilleurs.max()) == att else 'bitte prüfen')
        plt.step(seuils, c / c.max(), where='post', label=f'k = {k}')
    plt.xlabel('Schwellenwert'); plt.ylabel('Kosten (geteilt durch ihr Maximum)'); plt.legend(fontsize=8); plt.grid(alpha=0.4)
    plt.show()
f1 = []
for s in seuils:
    sig = p_exact >= s
    vp, fp, fn = (sig & est_spam).sum(), (sig & ~est_spam).sum(), (~sig & est_spam).sum()
    f1.append(2 * vp / (2 * vp + fp + fn))
f1 = np.array(f1)
best = seuils[np.isclose(f1, f1.max())]
print(f'maximaler F1 {f1.max():.2f}, für die Schwellenwerte von {best.min():.2f} bis {best.max():.2f}')
print('M6: exakter Wert', round(float(p_exact[5]), 4), '→ markiert beim Schwellenwert 0,27?', bool(p_exact[5] >= 0.27))

### c) bis e) Ihre Interpretationen

*Ihre Antworten:* …

## Aufgabe 5.5 — Vorhersagen, dann prüfen: eine Rundung am Rand des Schwellenwerts

*Begriffe: Entscheidungsregel (Gleichung 5.1), Sigmoidfunktion (Gleichung 5.4), F1 (Gleichung 5.8).* Gleiche Gewichte, aber eine Konstante b = −4,01.

### a) bis c) Ihre Vorhersagen und Berechnungen

*Ihre Vorhersage:* …

In [ ]:
# ZU ERGÄNZEN: die exakte Wahrscheinlichkeit von M3 mit b = −4,01 und ihre Entscheidung beim Schwellenwert 0,5
p_m3 = None            # eine Zahl (verwenden Sie sigmoide)
signale_m3 = None      # True oder False

In [ ]:
# Überprüfung, dann d) Entscheidungen auf den exakten und auf den gerundeten Werten
if p_m3 is None or signale_m3 is None:
    print('noch nicht ausgefüllt. Ergänzen Sie die vorherige Zelle.')
else:
    print(f'M3: p̂ = {p_m3:.4f} (angezeigt {p_m3:.2f}), markiert: {signale_m3} →',
          'richtig' if abs(p_m3 - 0.4975) < 0.0001 and signale_m3 is False else 'bitte prüfen')
p_coll = sigmoide(X @ w - 4.01)
for seuil in (0.5, 0.25):
    exact, arrondi = p_coll >= seuil, p_coll.round(2) >= seuil
    diff = list(douze['message'][exact != arrondi])
    print(f'Schwellenwert {seuil}: unterschiedliche Entscheidungen zwischen exaktem und gerundetem Wert: {diff if diff else "keine"}')
sig = p_coll >= 0.5
vp, fp, fn = int((sig & est_spam).sum()), int((sig & ~est_spam).sum()), int((~sig & est_spam).sum())
print(f'Kollege, Schwellenwert 0,5: VP = {vp}, FN = {fn}, FP = {fp}; Präzision = {vp / (vp + fp):.2f}, '
      f'Trefferquote = {vp / (vp + fn):.2f}, F1 = {2 * vp / (2 * vp + fp + fn):.2f}')

## Aufgabe 5.6 — Herausforderung: den logarithmischen Verlust hinabsteigen

*Begriffe: mittlerer Verlust (Gleichung 5.7), partielle Ableitungen und Gradient (Gleichung 3.6, Lektion 3.4), Sigmoidfunktion (Gleichung 5.4).*

### a) bis c) Auf Papier

*Ihre Herleitungen und Berechnungen:* …

In [ ]:
# ZU ERGÄNZEN: Gradient des mittleren Verlusts der zwölf Nachrichten (Frage b)
def gradient(w_, b_):
    """Gibt (dL/dw, dL/db) zurück: dL/dw ist ein Array aus zwei Zahlen."""
    return None   # ZU ERGÄNZEN: verwenden Sie X, y und sigmoide

In [ ]:
# Überprüfung, dann d) 1000 Gradientenschritte (η = 0,5) ab w = (1; 0,5), b = −4
def perte_moy(w_, b_):
    p_ = sigmoide(X @ w_ + b_)
    return float(np.mean(-(y * np.log(p_) + (1 - y) * np.log(1 - p_))))

if gradient(w, b) is None:
    print('noch nicht ausgefüllt. Ergänzen Sie die Funktion.')
else:
    gw, gb = gradient(w, b)
    print(f'zu Beginn: dL/db = {gb:.4f}, dL/dw = {np.round(gw, 4)} →',
          'richtig' if abs(gb - 0.0062) < 0.0001 and np.allclose(gw, [0.0144, -0.2368], atol=0.0001) else 'bitte prüfen')
    w_, b_ = w.astype(float), float(b)
    for pas in range(1, 1001):
        gw, gb = gradient(w_, b_)
        w_, b_ = w_ - 0.5 * gw, b_ - 0.5 * gb
        if pas in (1, 10, 100, 1000):
            print(f'nach {pas:>4} Schritten: Verlust = {perte_moy(w_, b_):.4f}, w = {np.round(w_, 2)}, b = {b_:.2f}')
    print('Endverlust →', 'richtig' if abs(perte_moy(w_, b_) - 0.0308) < 0.0005 else 'bitte prüfen')
# e) Gewichte, die die zwölf Nachrichten perfekt trennen: verdächtige Wörter ≥ 3 bei Spams, ≤ 2 bei legitimen
for facteur in (1, 10):
    print(f'w = (0 ; {2 * facteur}), b = {-5 * facteur}: mittlerer Verlust = {perte_moy(np.array([0, 2.0]) * facteur, -5.0 * facteur):.6f}')

### d) und e) Ihre Interpretationen

*Ihre Antworten:* …